In [1]:
import numpy as np
import pandas as pd
import pickle
import clip
import requests
from PIL import Image as PILImage
from io import BytesIO
import torch
from IPython.display import Image, display
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
import math

In [2]:
!pip install git+https://github.com/openai/CLIP.git --quiet

In [3]:
# Use a pipeline as a high-level helper
from transformers import pipeline

text_embedding_pipe = pipeline("feature-extraction", model="BAAI/bge-small-en-v1.5", device = -1)
# with open('sentiment_pipeline.pk', 'rb') as f:
#     sentiment_pipe = pickle.load(f)
sentiment_pipe = pipeline("text-classification", model="cardiffnlp/twitter-roberta-base-sentiment", device = -1, truncation = True, max_length = 512)
with open('ocr.pk', 'rb') as f:
    ocr_pipe = pickle.load(f)

model, preprocess = clip.load("ViT-B/32", device = "cpu")

Device set to use cpu
Device set to use cpu


In [4]:
def ocr_img_from_url(img_url):
    try:
        response = requests.get(img_url, stream=True)
        if response.status_code != 200:
            print(f"Failed to fetch: {img_url} (Status: {response.status_code})")
            return ""

        # Read image as bytes
        img_bytes = response.content
        
        # Use EasyOCR directly with bytes
        result = ocr_pipe.readtext(img_bytes)

        # Extract text with high confidence
        text_result = " ".join(text for _, text, prob in result if prob > 0.5)

        return text_result if text_result else ""

    except Exception as e:
        print(f"Error processing {img_url}: {e}")
        return ""
        
def img_embed_from_url(image_url):
    # Sample image and text data
    response = requests.get(image_url)
    
    # Preprocess image
    image = preprocess(
        PILImage.open(
            BytesIO(response.content)
        )
    ).unsqueeze(0).to("cpu")

    with torch.no_grad():
        image_features = model.encode_image(image)
    image_features /= image_features.norm(dim = -1, keepdim = True)
    return image_features[0]

def text_embed_from_str(text):
    text = str(text)
    embed = text_embedding_pipe(text)[0][0]
    return embed

def get_sentiment(text):
    text = str(text)
    res = sentiment_pipe(text)[0]['label']
    if res == 'LABEL_2':
        return 1
    elif res == 'LABEL_1':
        return 0
    else:
        return -1

In [5]:
def preprocess_obj (obj):
    blank_text_embed = text_embed_from_str('')
    blank_img_embed = img_embed_from_url('https://static.vecteezy.com/system/resources/thumbnails/012/680/916/small/blank-black-cement-wall-texture-for-background-with-copy-space-for-design-free-photo.jpg')
    
    columns_to_normalise = ['totalVolume', 'volume24h', 'marketCap', 'uniqueHolders', 'transferCount']
    obj[columns_to_normalise] = scaler.transform(pd.DataFrame([obj[columns_to_normalise]]))[0]
    
    if obj['name'] is not np.nan:
        obj['name_sentiment'] = get_sentiment(obj['name'])
        obj['name_embed'] = text_embed_from_str(obj['name'])
    elif obj['name'] is np.nan:
        obj['name_sentiment'] = 0
        obj['name_embed'] = blank_text_embed
    
    if obj['description'] is not np.nan:
        obj['description_sentiment'] = get_sentiment(obj['description'])
        obj['description_embed'] = text_embed_from_str(obj['description'])
    elif obj['description'] is np.nan:
        obj['description_sentiment'] = 0
        obj['description_embed'] = blank_text_embed

    # IMPORTANT!!!
    # Change the name to the corresponding image cdn column
    if obj['previewImageUrl'] is not np.nan:
        obj['img_embed'] = img_embed_from_url(obj['previewImageUrl'])
        obj['img_ocr'] = ocr_img_from_url(obj['previewImageUrl'])
    elif obj['previewImageUrl']is np.nan:
        obj['img_embed'] = blank_img_embed
        obj['img_ocr'] = ''

    if obj['img_ocr'] == '':
        obj['img_text_embed'] = blank_text_embed
        obj['img_text_sentiment'] = 0
    
    return obj
    
def preprocess_df (df):
    blank_text_embed = text_embed_from_str('')
    
    blank_img_embed = img_embed_from_url('https://static.vecteezy.com/system/resources/thumbnails/012/680/916/small/blank-black-cement-wall-texture-for-background-with-copy-space-for-design-free-photo.jpg')
    
    columns_to_normalise = ['totalVolume', 'volume24h', 'marketCap', 'uniqueHolders', 'transferCount']
    df[columns_to_normalise] = scaler.fit_transform(df[columns_to_normalise])
    
    df[['name', 'description']] = df[['name', 'description']].fillna('')

    df['name_sentiment'] = df['name'].apply(lambda x: get_sentiment(x) if x != '' else 0 )
    df['name_embed'] = df['name'].apply(lambda x: text_embed_from_str(x) if x != '' else blank_text_embed)
    
    df['description_sentiment'] = df['description'].apply(lambda x: get_sentiment(x) if x != '' else 0 )
    df['description_embed'] = df['description'].apply(lambda x: text_embed_from_str(x) if x != '' else blank_text_embed)

    df['img_embed'] = df['mediaPreviewUrl'].apply(lambda x: img_embed_from_url(x) if x is not np.nan else blank_img_embed)
    df['img_ocr'] = df['mediaPreviewUrl'].apply(lambda x: ocr_img_from_url(x) if x is not np.nan else '')
    df['img_text_sentiment'] = df['img_ocr'].apply(lambda x: get_sentiment(x) if x != '' else 0 )
    df['img_text_embed'] = df['img_ocr'].apply(lambda x: text_embed_from_str(x) if x != '' else blank_text_embed)

    # Convert timestamps to numerical values
    df['createdAt'] = pd.to_datetime(df['createdAt'])
    T = df['createdAt'].max()  # Latest timestamp
    
    time_diff = (T - df['createdAt']).dt.total_seconds()
    df['time_weight'] = np.exp(-0.001 * (time_diff))

    return df
    
        

In [6]:
df_big = pd.read_csv('Coin.csv')
df_coin = pd.read_csv('Coin_new.csv')

In [7]:
'0x0400436a9122d8be02d648fb7c68382c665963a9' in df_coin.address.values

False

In [8]:
example_post = df_big.loc[0].copy()

In [9]:
%%time
df_coin_preprocess = preprocess_df(df_coin)

Error processing https://media.decentralized-content.com/-/rs:fit:600:600/f:best/aHR0cHM6Ly9tYWdpYy5kZWNlbnRyYWxpemVkLWNvbnRlbnQuY29tL2lwZnMvYmFmeWJlaWVtNmx1eTRwbHNwY3c1YW82bnJ0ZWIyN2t4eTdyazVzYXBnYjVlczQ3eG42d2ltaHZrZ3U=: OpenCV(4.11.0) /Users/xperience/GHA-Actions-OpenCV/_work/opencv-python/opencv-python/opencv/modules/imgproc/src/color.cpp:199: error: (-215:Assertion failed) !_src.empty() in function 'cvtColor'

Error processing https://media.decentralized-content.com/-/rs:fit:600:600/f:best/aHR0cHM6Ly9tYWdpYy5kZWNlbnRyYWxpemVkLWNvbnRlbnQuY29tL2lwZnMvYmFmeWJlaWd1NTcycno1dXo3dnRpNGJtZDJsdHBzcG5ydGh0NHo2emk3cXd1ZGp1NGZpMmJldWx0emE=: OpenCV(4.11.0) /Users/xperience/GHA-Actions-OpenCV/_work/opencv-python/opencv-python/opencv/modules/imgproc/src/color.cpp:199: error: (-215:Assertion failed) !_src.empty() in function 'cvtColor'

Error processing https://media.decentralized-content.com/-/rs:fit:600:600/f:best/aHR0cHM6Ly9tYWdpYy5kZWNlbnRyYWxpemVkLWNvbnRlbnQuY29tL2lwZnMvYmFmeWJlaWh4dzZicWdtZ

In [10]:
%%time 
example_post_preprocess = preprocess_obj(example_post)

CPU times: user 3.24 s, sys: 1.18 s, total: 4.42 s
Wall time: 2.07 s


In [11]:
df_coin_preprocess

,id,name,symbol,description,createdAt,creatorAddress,uniqueHolders,mediaMimeType,totalSupply,totalVolume,...,updatedAt,name_sentiment,name_embed,description_sentiment,description_embed,img_embed,img_ocr,img_text_sentiment,img_text_embed,time_weight
0,034bc66f-344d-4f96-bf9f-ed7596a1edc9,Snake vibes,Snake vibes,,2025-02-23 17:35:27,0xc6c3fd75259eedb377a9e50bacb8bfb8a3a7db64,0.000036,image/jpeg,1000000000,2.866591e-08,...,2025-04-05 18:11:40.272,0,"[-0.442828506231308, -0.11557508260011673, 0.2...",0,"[0.4571411609649658, -0.012226441875100136, 0....","[tensor(-0.0099), tensor(-0.0040), tensor(-0.0...",,0,"[0.4571411609649658, -0.012226441875100136, 0....",0.000000e+00
1,03a3e278-a88c-4577-8194-0e08cb25816b,GkendhcXcAAKShq,GkendhcXcAAKShq,Sweet,2025-02-23 17:11:49,0x708e6e5eb3fa26e96c2eda0d987eb2a73b7c0217,0.000072,image/jpeg,1000000000,2.605992e-08,...,2025-04-05 18:11:25.755,0,"[-0.016978532075881958, 0.2147645503282547, 0....",1,"[-0.7258933186531067, -0.6866877675056458, 0.4...","[tensor(-0.0252), tensor(-0.0457), tensor(0.00...",By: Sadeghi Aeza @irancarnivoresconservation,0,"[-0.12997883558273315, 0.9504733681678772, -0....",0.000000e+00
2,044f44a9-d8b8-46a8-91b1-9932f352f66c,Hufflepuff,Hufflepuff,,2025-03-05 03:47:25,0xd36d589d2f626927f1a630277582d58890290c87,0.001657,image/jpeg,1000000000,7.755918e-04,...,2025-04-05 18:09:43.339,0,"[-0.4927580654621124, 0.009611759334802628, 0....",0,"[0.4571411609649658, -0.012226441875100136, 0....","[tensor(-0.0096), tensor(-0.0040), tensor(-0.0...",,0,"[0.4571411609649658, -0.012226441875100136, 0....",0.000000e+00
3,0534a108-44ab-4a14-8ae6-8125841731a4,"3/18/25, 11:02 PM","3/18/25, 11:02 PM",,2025-03-19 03:02:57,0x5e384ba3ed638eb7e17db46ca75e5264d061c719,0.013468,image/jpeg,1000000000,8.678847e-04,...,2025-04-05 14:07:25.304,0,"[-0.39407679438591003, -0.16728772222995758, -...",0,"[0.4571411609649658, -0.012226441875100136, 0....","[tensor(-0.0073), tensor(-0.0079), tensor(0.02...","can someone explain to me, very slowly, how mo...",0,"[-0.33836063742637634, -0.3991420567035675, -0...",0.000000e+00
4,0920150a-5df1-427f-96a7-eb083befbc35,grey,grey,,2025-02-23 23:36:53,0xb61c855096311c0bde071c6dfff6e3bc121c738a,0.000072,image/png,1000000000,1.607984e-05,...,2025-04-05 18:09:54.534,0,"[-0.24905739724636078, -0.7531082034111023, 0....",0,"[0.4571411609649658, -0.012226441875100136, 0....","[tensor(-0.0062), tensor(-0.0122), tensor(-0.0...",,0,"[0.4571411609649658, -0.012226441875100136, 0....",0.000000e+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
158,fbfef138-2490-4de1-bca6-3804c91f1591,Sabzeh,Sabzeh,Sizdeh Bedar Cast-off,2025-04-04 23:09:35,0x94b6f485246f584098b9516c5f5a3404bd690a1f,0.000252,image/jpeg,1000000000,4.990822e-05,...,2025-04-05 14:03:39.048,0,"[0.07695156335830688, 0.41401344537734985, -0....",0,"[0.08597046136856079, 0.7525924444198608, -0.2...","[tensor(-0.0431), tensor(-0.0008), tensor(0.01...",,0,"[0.4571411609649658, -0.012226441875100136, 0....",1.103279e-21
159,fc2c1b28-606e-4dbf-91c9-2e64b20e2032,The ambisius $Zero,The ambisius $Zero,$Zero scary movie,2025-02-23 07:39:19,0x3941a9af09d260e80e5f999ebf1d29127477d382,0.000000,image/gif,1000000000,2.432259e-08,...,2025-04-05 18:11:17.004,0,"[-0.7605044841766357, -0.18821880221366882, -0...",-1,"[-0.3407902717590332, -0.20230664312839508, -0...","[tensor(0.0254), tensor(0.0238), tensor(0.0042...",,0,"[0.4571411609649658, -0.012226441875100136, 0....",0.000000e+00
160,fc30d7e6-937a-4e59-9969-b6dd093816d2,Tower,Tower,,2025-02-23 14:23:41,0xc947c209fb72ea759b1be12a06cbc9c65dbe798f,0.000036,image/jpeg,1000000000,8.686639e-10,...,2025-04-05 18:12:11.547,0,"[-0.3612920939922333, -0.5669062733650208, 0.3...",0,"[0.4571411609649658, -0.012226441875100136, 0....","[tensor(-0.0072), tensor(-0.0198), tensor(0.01...",,0,"[0.4571411609649658, -0.012226441875100136, 0....",0.000000e+00
161,fccb60a5-958d-46d2-aaed-272f33e0aeaa,$VB,$VB,coin more\n\nby velvetblue,2025-04-04 22:43:33,0x96e41e

In [12]:
df_coin_preprocess.columns

Index(['id', 'name', 'symbol', 'description', 'createdAt', 'creatorAddress',
       'uniqueHolders', 'mediaMimeType', 'totalSupply', 'totalVolume',
       'volume24h', 'marketCap', 'address', 'mediaPreviewUrl',
       'marketCapDelta24h', 'mediaOriginalUri', 'scrapedAt', 'transferCount',
       'updatedAt', 'name_sentiment', 'name_embed', 'description_sentiment',
       'description_embed', 'img_embed', 'img_ocr', 'img_text_sentiment',
       'img_text_embed', 'time_weight'],
      dtype='object')

In [13]:
example_post_preprocess

id                              R3JhcGhRTFpvcmEyMFRva2VuOkJBU0UtTUFJTk5FVC4weD...
address                                0x0400436a9122d8be02d648fb7c68382c665963a9
name                                                                    Cherepaxa
symbol                                                                  Cherepaxa
description                                                                   NaN
totalSupply                                                            1000000000
totalVolume                                                                   0.0
volume24h                                                                0.000012
createdAt                                                     2025-03-15 06:56:45
creatorAddress                         0xd0db2fe525f49f59e29fc2e69765572b14e3c1bd
marketCap                                                                     0.0
marketCapDelta24h                                                            -1.0
chainId         

In [14]:
def sentiment_calculator(sen1, sen2):
    distance = abs(sen1 - sen2)
    if distance == 0:
        return 1
    elif distance == 1:
        return 0.5
    else:
        return 0
        
def calculate_sim(obj1, obj2):
    financial_columns = ['totalVolume', 'volume24h', 'marketCap', 'uniqueHolders', 'transferCount']
    w_sentiment = 0.15
    w_financial = 0.15
    w_embed = 0.1
    
    sen_ocr = sentiment_calculator(obj1['img_text_sentiment'], obj2['img_text_sentiment']) 
    sen_name = sentiment_calculator(obj1['name_sentiment'],obj2['name_sentiment'])
    sen_description = sentiment_calculator(obj1['description_sentiment'],obj2['description_sentiment']) 

    img_embed = cosine_similarity([obj1['img_embed']], [obj2['img_embed']])[0][0]
    name_embed = cosine_similarity([obj1['name_embed']], [obj2['name_embed']])[0][0]
    description_embed = cosine_similarity([obj1['description_embed']], [obj2['description_embed']])[0][0]
    img_text_embed = cosine_similarity([obj1['description_embed']], [obj2['description_embed']])[0][0]
    
    financial_sim = cosine_similarity([obj1[financial_columns]], [obj2[financial_columns]])[0][0]
    sentiment_sim = sen_ocr + sen_name + sen_description
    embed_sim = img_embed + name_embed + description_embed + img_text_embed
    
    total_distance = w_sentiment * (sentiment_sim) + w_embed * (embed_sim)  + w_financial * (financial_sim)
    return total_distance 

In [15]:
calculate_sim (example_post_preprocess, df_coin_preprocess.loc[0])

0.7653919625834483

In [16]:
df_coin_preprocess['similarity'] = df_coin_preprocess.apply(lambda x: calculate_sim(example_post_preprocess, x), axis = 1)

In [17]:
weighted_mean_similarity = np.average(df_coin_preprocess['similarity'], weights=df_coin_preprocess['time_weight'])

In [18]:
weighted_mean_similarity

0.7265958906217348

In [19]:
df_coin_preprocess['similarity'].describe()

count    163.000000
mean       0.732063
std        0.095383
min        0.420210
25%        0.670162
50%        0.741608
75%        0.807229
max        0.912205
Name: similarity, dtype: float64

In [20]:
import pickle
with open ('sentiment_pipeline.pk', 'wb') as f:
    pickle.dump(sentiment_pipe, f)
with open('text_embedding_pipeline.pk', 'wb') as g:
    pickle.dump(text_embedding_pipe, g)
with open('img_embedding_model.pk', 'wb') as h:
    pickle.dump(model, h)
with open('img_embedding_preprocess.pk', 'wb') as i:
    pickle.dump(preprocess, i)